In [5]:
%%capture
!pip install gradio
!pip install transformers
!pip install torch
!pip install langchain
!pip install langchain_community

##**IBM Granite 3.2: Reasoning, vision, forecasting and more**

Granite-3.2-2B-Instruct is an 2-billion-parameter, long-context AI model fine-tuned for thinking capabilities. Built on top of Granite-3.1-2B-Instruct, it has been trained using a mix of permissively licensed open-source datasets and internally generated synthetic data designed for reasoning tasks.

In [6]:
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
import torch

##Load the model, tockenizer, and pipeline from hugginface

In [7]:
tokenizer = AutoTokenizer.from_pretrained("ibm-granite/granite-3.2-2b-instruct")

model = AutoModelForCausalLM.from_pretrained("ibm-granite/granite-3.2-2b-instruct",
                                             torch_dtype=torch.float16, #torch type = FP16 for qunatiation and faster infrence time while using lower memeor amount.
                                             device_map="auto")

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_length=500)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Device set to use cpu


In [8]:
llm = HuggingFacePipeline(pipeline=pipe)

/tmp/ipython-input-2691217058.py:1: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [9]:
#Knowledge Base and prompt template
template = """
Generate a {duration}-day travel itinerary for {destination}
"""
prompt = PromptTemplate(input_variables=["destination","duration"], template=template)

In [10]:
chain = LLMChain(llm=llm, prompt=prompt)

/tmp/ipython-input-1305865249.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt)


In [12]:
def generate_travel_schedule(destination, duration):
    """
    Generates a travel itinerary using a LangChain LLMChain.

    Args:
        destination (str): The desired travel destination.
        duration (str): The duration of the trip (e.g., "3-day", "week-long").

    Returns:
        str: The generated travel itinerary text.
    """
    # Uses the previously defined LangChain LLMChain to generate a travel itinerary
    # based on the provided destination and duration.
    return chain.run({"destination": destination, "duration":duration})

##Gradio, Simple Chatbot Interface for our Customer Service


In [13]:
with gr.Blocks() as demo:
    gr.Markdown("AI Travel Planner with IBM Granite")
    destination_input = gr.Textbox(label="Destination", placeholder="e.g., Paris")
    duration_input = gr.Number(label="Days", value=3)
    submit_button = gr.Button("Generate Itinerary")
    output_box = gr.Textbox(label="Itinerary", interactive=False, lines=15)
    submit_button.click(fn=generate_travel_schedule, inputs=[destination_input, duration_input], outputs=output_box)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d8df1cfb691a7e93f2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Generate a 2-day travel itinerary for Paris 😆

**Day 1: Arrival & Explore the Heart of Paris**

*Morning:*
- Begin your Parisian adventure at the iconic **Eiffel Tower**. Take the elevator to the second floor for panoramic city views.
- Visit the **Louvre Museum**, home to thousands of works of art, including the Mona Lisa. Book your tickets online to avoid long queues.
- Stroll along the **Champs-Élysées**, an iconic avenue famous for its annual Bastille Day parade and various shops and cafés.

*Afternoon:*
- Head to the **Notre-Dame Cathedral** and admire its Gothic architecture. Note the current restoration work.
- Explore the charming **Saint-Germain-des-Prés** neighborhood, known for its bohemian history, cafés, and bookstores.

*Evening:*
- Dine at a traditional French bistro in the **Latin Quarter** (Quartier Latin), like Chez L'Ami Jean, enjoying classic dishes like coq au vin or escargot.
- Take a leisurely evening stroll along the **Seine River**, possibly ending at the **Musée d'Orsay** to admire its impressive collection of Impressionist and Post-Impressionist masterpieces.

**Day 2: Artistic Districts & Gourmet Indulgence**

*Morning:*
- Begin your day with a visit to **Montmartre**, the former home of artists like Picasso and Van Gogh.
- Climb the steps to the majestic **Sacré-Cœur Basilica** for a rewarding panorama of Paris.
- Explore the vibrant **Place du Tertre**, where local artists paint while visitors enjoy the atmosphere.

*Afternoon:*
- Head to the **Palais Garnier**, an opulent Beaux-Arts building that's now a renowned opera house, complete with a spectacular foyer